# ✦ ASSESSLY — AI Hiring Agent
### Transform job requirements into validated hiring decisions

**Workflow:** Create Job → AI Generates Assessment → Candidate Takes Test → AI Evaluates → Hiring Recommendation

In [ ]:
import streamlit as st
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

## 📊 Dashboard

In [ ]:
jobs_count = session.sql("SELECT COUNT(*) as cnt FROM ASSESSLY_DB.ASSESSLY_SCHEMA.JOBS").collect()[0]["CNT"]
assessments_count = session.sql("SELECT COUNT(*) as cnt FROM ASSESSLY_DB.ASSESSLY_SCHEMA.ASSESSMENTS WHERE status = 'PUBLISHED'").collect()[0]["CNT"]
candidates_count = session.sql("SELECT COUNT(*) as cnt FROM ASSESSLY_DB.ASSESSLY_SCHEMA.CANDIDATES").collect()[0]["CNT"]
evaluated_count = session.sql("SELECT COUNT(*) as cnt FROM ASSESSLY_DB.ASSESSLY_SCHEMA.CANDIDATES WHERE status = 'EVALUATED'").collect()[0]["CNT"]

col1, col2, col3, col4 = st.columns(4)
col1.metric('Total Jobs', jobs_count)
col2.metric('Active Assessments', assessments_count)
col3.metric('Total Candidates', candidates_count)
col4.metric('Evaluated', evaluated_count)

---
## 📝 Create New Job & Generate Assessment
Fill in the job details below. AI will automatically analyze competencies and generate assessment questions.

In [ ]:
with st.form('create_job_form'):
    title = st.text_input('Job Title', placeholder='e.g. Senior Backend Engineer')
    col1, col2, col3 = st.columns(3)
    with col1:
        department = st.text_input('Department', placeholder='Engineering')
    with col2:
        seniority = st.selectbox('Seniority', ['Junior', 'Mid-level', 'Senior', 'Lead', 'Principal'])
    with col3:
        employment_type = st.selectbox('Type', ['Full-time', 'Part-time', 'Contract'])
    description = st.text_area('Job Description', height=100, placeholder='Describe the role...')
    requirements = st.text_area('Requirements & Skills', height=100, placeholder='Python, FastAPI, PostgreSQL...')
    auto_generate = st.checkbox('Auto-generate assessment', value=True)
    submitted = st.form_submit_button('Create Job & Generate Assessment', type='primary', use_container_width=True)

if submitted and title and description:
    with st.spinner('Creating job...'):
        session.sql("INSERT INTO ASSESSLY_DB.ASSESSLY_SCHEMA.JOBS (title, description, requirements, department, seniority_level, employment_type, status) VALUES (?, ?, ?, ?, ?, ?, 'DRAFT')", params=[title, description, requirements, department, seniority, employment_type]).collect()
        job_id = session.sql('SELECT MAX(job_id) as id FROM ASSESSLY_DB.ASSESSLY_SCHEMA.JOBS').collect()[0]['ID']
    st.success(f'Job **{title}** created! (ID: {job_id})')
    if auto_generate:
        with st.spinner('AI is mapping competencies...'):
            r1 = session.sql(f'CALL ASSESSLY_DB.ASSESSLY_SCHEMA.GENERATE_COMPETENCIES({job_id})').collect()[0][0]
            st.info(r1)
        with st.spinner('AI is generating assessment questions (30-60s)...'):
            r2 = session.sql(f'CALL ASSESSLY_DB.ASSESSLY_SCHEMA.GENERATE_ASSESSMENT({job_id})').collect()[0][0]
            st.success(r2)
        st.balloons()

---
## 📋 Manage Assessments
Review AI-generated assessments, view questions, and publish.

In [ ]:
assessments_df = session.sql("""
    SELECT a.assessment_id, a.title, j.title as job_title, a.duration_minutes,
           a.total_questions, a.passing_score, a.status
    FROM ASSESSLY_DB.ASSESSLY_SCHEMA.ASSESSMENTS a
    JOIN ASSESSLY_DB.ASSESSLY_SCHEMA.JOBS j ON j.job_id = a.job_id
    ORDER BY a.created_at DESC
""").to_pandas()

if len(assessments_df) > 0:
    st.dataframe(assessments_df, use_container_width=True, hide_index=True)
    
    selected_id = st.selectbox('Select Assessment ID to manage', assessments_df['ASSESSMENT_ID'].tolist())
    
    col1, col2 = st.columns(2)
    if col1.button('Publish Assessment', type='primary'):
        session.sql(f"UPDATE ASSESSLY_DB.ASSESSLY_SCHEMA.ASSESSMENTS SET status = 'PUBLISHED', published_at = CURRENT_TIMESTAMP() WHERE assessment_id = {selected_id}").collect()
        st.success('Assessment published!')
    
    if col2.button('View Questions'):
        questions_df = session.sql(f"""
            SELECT sort_order as Q, question_type as Type, difficulty as Difficulty,
                   question_text as Question, max_score as Points
            FROM ASSESSLY_DB.ASSESSLY_SCHEMA.QUESTIONS
            WHERE assessment_id = {selected_id}
            ORDER BY sort_order
        """).to_pandas()
        st.dataframe(questions_df, use_container_width=True, hide_index=True)
else:
    st.info('No assessments yet. Create a job above to generate one.')

---
## 👥 Manage Candidates
Add candidates and generate access tokens for them to take the assessment.

In [ ]:
published = session.sql("""
    SELECT a.assessment_id, a.title || ' - ' || j.title as label
    FROM ASSESSLY_DB.ASSESSLY_SCHEMA.ASSESSMENTS a
    JOIN ASSESSLY_DB.ASSESSLY_SCHEMA.JOBS j ON j.job_id = a.job_id
    WHERE a.status = 'PUBLISHED'
""").to_pandas()

if len(published) > 0:
    with st.form('add_candidate'):
        assess_id = st.selectbox('Assessment', published['ASSESSMENT_ID'].tolist(), format_func=lambda x: published[published['ASSESSMENT_ID']==x]['LABEL'].values[0])
        col1, col2 = st.columns(2)
        cand_name = col1.text_input('Candidate Name')
        cand_email = col2.text_input('Candidate Email')
        if st.form_submit_button('Add Candidate', type='primary', use_container_width=True):
            if cand_name:
                result = session.sql(f"CALL ASSESSLY_DB.ASSESSLY_SCHEMA.ADD_CANDIDATE({assess_id}, ?, ?)", params=[cand_name, cand_email]).collect()[0][0]
                st.success(result)
            else:
                st.error('Name is required')
else:
    st.warning('Publish an assessment first.')

st.subheader('Candidate List')
candidates_df = session.sql("""
    SELECT c.candidate_id as ID, c.candidate_name as Name, c.candidate_email as Email,
           c.access_token as Token, c.status as Status, a.title as Assessment
    FROM ASSESSLY_DB.ASSESSLY_SCHEMA.CANDIDATES c
    JOIN ASSESSLY_DB.ASSESSLY_SCHEMA.ASSESSMENTS a ON a.assessment_id = c.assessment_id
    ORDER BY c.created_at DESC
""").to_pandas()
if len(candidates_df) > 0:
    st.dataframe(candidates_df, use_container_width=True, hide_index=True)
else:
    st.info('No candidates yet.')

---
## 🎯 Candidate Assessment Portal
Candidates enter their access token to take the assessment.

In [ ]:
token = st.text_input('Enter Access Token', type='password', placeholder='Paste access token here')

if st.button('Load Assessment', type='primary') and token:
    candidate = session.sql(f"""
        SELECT c.candidate_id, c.candidate_name, c.assessment_id, c.status, a.title
        FROM ASSESSLY_DB.ASSESSLY_SCHEMA.CANDIDATES c
        JOIN ASSESSLY_DB.ASSESSLY_SCHEMA.ASSESSMENTS a ON a.assessment_id = c.assessment_id
        WHERE c.access_token = '{token}' AND a.status = 'PUBLISHED'
    """).collect()
    
    if not candidate:
        st.error('Invalid token.')
    elif candidate[0]['STATUS'] in ('COMPLETED', 'EVALUATED'):
        st.warning('Assessment already completed.')
    else:
        c = candidate[0]
        st.session_state['cand_id'] = c['CANDIDATE_ID']
        st.session_state['assess_id'] = c['ASSESSMENT_ID']
        session.sql(f"UPDATE ASSESSLY_DB.ASSESSLY_SCHEMA.CANDIDATES SET status = 'IN_PROGRESS', started_at = CURRENT_TIMESTAMP() WHERE candidate_id = {c['CANDIDATE_ID']}").collect()
        st.success(f"Welcome **{c['CANDIDATE_NAME']}**! Assessment: {c['TITLE']}")

if 'cand_id' in st.session_state:
    cand_id = st.session_state['cand_id']
    assess_id = st.session_state['assess_id']
    questions = session.sql(f"""
        SELECT question_id, question_type, question_text, options, max_score, sort_order
        FROM ASSESSLY_DB.ASSESSLY_SCHEMA.QUESTIONS WHERE assessment_id = {assess_id} ORDER BY sort_order
    """).to_pandas()
    
    answers = {}
    with st.form('answer_form'):
        for _, row in questions.iterrows():
            st.markdown(f"**Q{row['SORT_ORDER']}.** ({row['QUESTION_TYPE']} - {row['MAX_SCORE']} pts)")
            st.write(row['QUESTION_TEXT'])
            if row['QUESTION_TYPE'] == 'MCQ' and row['OPTIONS']:
                opts = json.loads(row['OPTIONS']) if isinstance(row['OPTIONS'], str) else row['OPTIONS']
                if opts:
                    answers[row['QUESTION_ID']] = st.radio('Answer:', opts, key=f"q_{row['QUESTION_ID']}")
            elif row['QUESTION_TYPE'] == 'SHORT_ANSWER':
                answers[row['QUESTION_ID']] = st.text_area('Answer:', height=80, key=f"q_{row['QUESTION_ID']}")
            else:
                answers[row['QUESTION_ID']] = st.text_area('Answer:', height=150, key=f"q_{row['QUESTION_ID']}")
            st.markdown('---')
        
        if st.form_submit_button('Submit Assessment', type='primary', use_container_width=True):
            for q_id, ans in answers.items():
                if ans:
                    session.sql(f"INSERT INTO ASSESSLY_DB.ASSESSLY_SCHEMA.CANDIDATE_ANSWERS (candidate_id, question_id, answer_text) VALUES ({cand_id}, {q_id}, ?)", params=[ans]).collect()
            session.sql(f"UPDATE ASSESSLY_DB.ASSESSLY_SCHEMA.CANDIDATES SET status = 'COMPLETED', completed_at = CURRENT_TIMESTAMP() WHERE candidate_id = {cand_id}").collect()
            del st.session_state['cand_id']
            del st.session_state['assess_id']
            st.success('Assessment submitted!')
            st.balloons()

---
## 🤖 AI Evaluation
Select a completed candidate and let AI evaluate their answers.

In [ ]:
completed_df = session.sql("""
    SELECT c.candidate_id, c.candidate_name, a.title as assessment
    FROM ASSESSLY_DB.ASSESSLY_SCHEMA.CANDIDATES c
    JOIN ASSESSLY_DB.ASSESSLY_SCHEMA.ASSESSMENTS a ON a.assessment_id = c.assessment_id
    WHERE c.status = 'COMPLETED'
""").to_pandas()

if len(completed_df) > 0:
    eval_options = {f"{row['CANDIDATE_NAME']} — {row['ASSESSMENT']}": row['CANDIDATE_ID'] for _, row in completed_df.iterrows()}
    selected_cand = st.selectbox('Select candidate to evaluate', list(eval_options.keys()))
    
    if st.button('Run AI Evaluation', type='primary'):
        cid = eval_options[selected_cand]
        with st.spinner('AI is evaluating answers... (30-60 seconds)'):
            result = session.sql(f'CALL ASSESSLY_DB.ASSESSLY_SCHEMA.EVALUATE_CANDIDATE({cid})').collect()[0][0]
            st.success(result)
else:
    st.info('No completed assessments to evaluate.')

---
## 📊 Results & Hiring Recommendations

In [ ]:
results_df = session.sql("""
    SELECT c.candidate_name as Candidate, j.title as Position,
           e.percentage_score as Score, e.recommendation as Recommendation,
           e.ai_reasoning as AI_Assessment, e.competency_scores
    FROM ASSESSLY_DB.ASSESSLY_SCHEMA.EVALUATIONS e
    JOIN ASSESSLY_DB.ASSESSLY_SCHEMA.CANDIDATES c ON c.candidate_id = e.candidate_id
    JOIN ASSESSLY_DB.ASSESSLY_SCHEMA.ASSESSMENTS a ON a.assessment_id = c.assessment_id
    JOIN ASSESSLY_DB.ASSESSLY_SCHEMA.JOBS j ON j.job_id = a.job_id
    ORDER BY e.percentage_score DESC
""").to_pandas()

if len(results_df) > 0:
    col1, col2, col3 = st.columns(3)
    col1.metric('Total Evaluated', len(results_df))
    col2.metric('Strong Hire', len(results_df[results_df['RECOMMENDATION'] == 'STRONG_HIRE']))
    col3.metric('No Hire', len(results_df[results_df['RECOMMENDATION'] == 'NO_HIRE']))
    
    st.subheader('Ranking')
    st.dataframe(results_df[['CANDIDATE', 'POSITION', 'SCORE', 'RECOMMENDATION']], use_container_width=True, hide_index=True)
    
    st.subheader('Detailed Report')
    for _, row in results_df.iterrows():
        with st.expander(f"{row['CANDIDATE']} — {row['SCORE']:.0f}% — {row['RECOMMENDATION']}"):
            st.write(row['AI_ASSESSMENT'])
            if row['COMPETENCY_SCORES']:
                scores = json.loads(row['COMPETENCY_SCORES']) if isinstance(row['COMPETENCY_SCORES'], str) else row['COMPETENCY_SCORES']
                if scores:
                    col1, col2 = st.columns(2)
                    with col1:
                        st.markdown('**Strengths:**')
                        for s in scores.get('strengths', []):
                            st.markdown(f'- ✅ {s}')
                    with col2:
                        st.markdown('**Weaknesses:**')
                        for w in scores.get('weaknesses', []):
                            st.markdown(f'- ⚠️ {w}')
else:
    st.info('No results yet. Evaluate candidates above.')

---
## 📄 Generate Detailed Hiring Report

In [ ]:
evaluated_df = session.sql("""
    SELECT c.candidate_id, c.candidate_name, j.title as position
    FROM ASSESSLY_DB.ASSESSLY_SCHEMA.CANDIDATES c
    JOIN ASSESSLY_DB.ASSESSLY_SCHEMA.ASSESSMENTS a ON a.assessment_id = c.assessment_id
    JOIN ASSESSLY_DB.ASSESSLY_SCHEMA.JOBS j ON j.job_id = a.job_id
    WHERE c.status = 'EVALUATED'
""").to_pandas()

if len(evaluated_df) > 0:
    report_options = {f"{row['CANDIDATE_NAME']} — {row['POSITION']}": row['CANDIDATE_ID'] for _, row in evaluated_df.iterrows()}
    selected_report = st.selectbox('Select candidate for report', list(report_options.keys()))
    
    if st.button('Generate Hiring Report', type='primary'):
        cid = report_options[selected_report]
        with st.spinner('Generating comprehensive report...'):
            report = session.sql(f'CALL ASSESSLY_DB.ASSESSLY_SCHEMA.GENERATE_RECOMMENDATION({cid})').collect()[0][0]
            st.markdown('### Hiring Recommendation Report')
            st.markdown(report)
else:
    st.info('Evaluate candidates first to generate reports.')